# **Brain CT Image Denoising: DCT vs Gaussian Filtering**


---


## Project Overview
This project compares the denoising performance of 8×8 block DCT with soft thresholding and Gaussian filtering on low-dose brain CT images.

## Key Methods
1. **Denoising Methods**
   - DCT Denoising: 8×8 block DCT + JPEG quantization table (scipy.fft.dctn, idctn, quantization rounding)
   - Gaussian Filtering: Weighted average with Gaussian kernel (skimage.filters.gaussian)

2. **Evaluation Metrics**
   - PSNR (Peak Signal-to-Noise Ratio)
   - SSIM (Structural Similarity Index)
   - Both from `skimage.metrics`

3. **Visualization**
   - Image comparison
   - PSNR/SSIM bar chart

## Data Source
The brain CT dataset used in this study is obtained from the PhysioNet repository:

Hssayeni, M. (2020). Computed Tomography Images for Intracranial Hemorrhage Detection and Segmentation. PhysioNet. https://physionet.org/content/ct-ich/1.3.1/

## Workspace

In [ ]:
pip install numpy matplotlib scipy scikit-image opencv-python pillow

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, filters, metrics
from scipy.fft import dctn, idctn
from skimage.util import img_as_float
import warnings

Step2. Load & Preprocess Image

In [ ]:
# 在 Colab 中运行此单元格
!git clone https://github.com/ziyiWANGG/7AAVDH26_001_Final_Coding_Project.git

# 查看克隆后的目录结构
!ls -la 7AAVDH26_001_Final_Coding_Project/

# 切换到项目目录（Colab 会自动保留当前路径）
import os
os.chdir('/content/7AAVDH26_001_Final_Coding_Project')
print(f"当前工作目录: {os.getcwd()}")

# 查看 CT_images 是否存在
!ls -la CT_images/

In [ ]:
base_path = "./CT_images"  # 或者使用完整路径
folders = ["054_brain", "079_brain"]

# ========== 收集所有图片 ==========
image_paths = []
for folder in folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.exists(folder_path):
        # 获取文件夹中所有 jpg 图片
        jpg_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.jpg')]
        for jpg_file in jpg_files:
            full_path = os.path.join(folder_path, jpg_file)
            image_paths.append({
                'path': full_path,
                'folder': folder,
                'filename': jpg_file
            })
        print(f"📁 {folder}: 找到 {len(jpg_files)} 张图片")
    else:
        print(f"❌ 文件夹不存在: {folder_path}")

print(f"\n总计找到 {len(image_paths)} 张图片")

# 显示前5张图片路径作为示例
for i, img_info in enumerate(image_paths[:5]):
    print(f"{i+1}. {img_info['folder']}/{img_info['filename']}")

**Step1**. DCT Denoising

In [ ]:
def dct_denoise(image, quality_factor=1.0):
    """
    对灰度图像进行分块 DCT + 量化 + 反量化 + IDCT 去噪
    image: 二维 numpy 数组，值范围 0~255
    quality_factor: 量化缩放因子（<1 增强去噪，>1 保留更多细节）
    """
    # JPEG 标准亮度量化表
    Q_table = np.array([
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99]
    ])
    Q_table = Q_table * quality_factor

    h, w = image.shape
    # 填充到 8 的倍数
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    img_pad = np.pad(image, ((0, pad_h), (0, pad_w)), mode='edge')
    img_pad = img_pad.astype(np.float32)

    denoised_pad = np.zeros_like(img_pad)
    for i in range(0, img_pad.shape[0], 8):
        for j in range(0, img_pad.shape[1], 8):
            block = img_pad[i:i+8, j:j+8]
            # 中心化
            block_centered = block - 128
            # DCT (正交归一化)
            dct_block = dctn(block_centered, norm='ortho')
            # 量化
            quantized = np.round(dct_block / Q_table)
            # 反量化
            dequantized = quantized * Q_table
            # IDCT
            idct_block = idctn(dequantized, norm='ortho')
            # 反中心化
            denoised_pad[i:i+8, j:j+8] = idct_block + 128
    # 裁剪回原尺寸
    denoised = np.clip(denoised_pad[:h, :w], 0, 255).astype(np.uint8)
    return denoised

**Step2**. Gaussian Filter Denoising

In [ ]:
def gaussian_denoise(image, sigma=1.0):
    """高斯滤波去噪，sigma 控制平滑强度"""
    image_float = img_as_float(image)  # 转为 [0,1] 浮点
    denoised_float = filters.gaussian(image_float, sigma=sigma, preserve_range=False)
    denoised = (denoised_float * 255).astype(np.uint8)
    return denoised

**Step3**. Calculate PSNR & SSIM

In [ ]:
def calculate_psnr(img_orig, img_denoised):
    """计算峰值信噪比"""
    mse = np.mean((img_orig.astype(np.float32) - img_denoised.astype(np.float32)) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr

def calculate_ssim(img_orig, img_denoised):
    """计算结构相似性指数（使用 skimage 内置）"""
    # 输入范围 0-255，需转为 float 并缩放到 [0,1]
    return metrics.structural_similarity(img_orig, img_denoised, data_range=255)

**Step4**. 指定图片路径

In [ ]:
# 路径配置（已在项目根目录）
base_path = "./CT_images"
folders = ["054_brain", "079_brain"]

# 收集所有 jpg 图片路径（包含详细信息）
image_paths = []
image_details = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.exists(folder_path):
        for fname in os.listdir(folder_path):
            if fname.lower().endswith('.jpg'):
                full_path = os.path.join(folder_path, fname)
                image_paths.append(full_path)
                image_details.append({
                    'path': full_path,
                    'folder': folder,
                    'filename': fname
                })
    else:
        print(f"警告：路径 {folder_path} 不存在，请检查 base_path 设置")

print(f"找到 {len(image_paths)} 张图片")
for i, detail in enumerate(image_details[:5]):
    print(f"{i+1}. {detail['folder']}/{detail['filename']}")

**Step5**. 对每张图片应用两种去噪方法并记录指标

In [ ]:
results = []  # 存储每张图片的结果

# 设置去噪参数（可自行调整）
dct_quality = 0.8      # DCT 量化因子，越小去噪越强
gaussian_sigma = 1.2    # 高斯滤波标准差

for img_path in image_paths:
    # 读取灰度图
    img_orig = io.imread(img_path, as_gray=True)
    if img_orig.dtype != np.uint8:
        img_orig = (img_orig * 255).astype(np.uint8)

    # 去噪
    img_dct = dct_denoise(img_orig, quality_factor=dct_quality)
    img_gauss = gaussian_denoise(img_orig, sigma=gaussian_sigma)

    # 计算指标（相对于原始图像，注意局限性见结论）
    psnr_dct = calculate_psnr(img_orig, img_dct)
    ssim_dct = calculate_ssim(img_orig, img_dct)
    psnr_gauss = calculate_psnr(img_orig, img_gauss)
    ssim_gauss = calculate_ssim(img_orig, img_gauss)

    results.append({
        'name': os.path.basename(img_path),
        'psnr_dct': psnr_dct, 'ssim_dct': ssim_dct,
        'psnr_gauss': psnr_gauss, 'ssim_gauss': ssim_gauss,
        'img_orig': img_orig, 'img_dct': img_dct, 'img_gauss': img_gauss
    })
    print(f"{os.path.basename(img_path)}: DCT (PSNR={psnr_dct:.2f}, SSIM={ssim_dct:.3f}) | Gauss (PSNR={psnr_gauss:.2f}, SSIM={ssim_gauss:.3f})")

显示最佳/最差案例

In [ ]:
# 找出 PSNR 最高和最低的图片
best_dct = max(results, key=lambda x: x['psnr_dct'])
worst_dct = min(results, key=lambda x: x['psnr_dct'])
best_gauss = max(results, key=lambda x: x['psnr_gauss'])
worst_gauss = min(results, key=lambda x: x['psnr_gauss'])

print("\n" + "="*60)
print("最佳/最差案例分析")
print("="*60)
print(f"\nDCT 方法最佳: {best_dct['folder']}/{best_dct['name']} (PSNR={best_dct['psnr_dct']:.2f} dB)")
print(f"DCT 方法最差: {worst_dct['folder']}/{worst_dct['name']} (PSNR={worst_dct['psnr_dct']:.2f} dB)")
print(f"\n高斯滤波最佳: {best_gauss['folder']}/{best_gauss['name']} (PSNR={best_gauss['psnr_gauss']:.2f} dB)")
print(f"高斯滤波最差: {worst_gauss['folder']}/{worst_gauss['name']} (PSNR={worst_gauss['psnr_gauss']:.2f} dB)")

**Step6**. 可视化：显示单张图片的处理结果示例

In [ ]:
# 由于 results 中没有 folder 信息，我们需要从 image_details 中获取文件夹信息
# 或者假设 results 的顺序与 image_paths 一致

# 方法1：假设 results 的顺序与 image_details 一致
# 重新构建包含文件夹信息的结果列表
results_with_folder = []
for idx, r in enumerate(results):
    r['folder'] = image_details[idx]['folder']  # 添加 folder 键
    results_with_folder.append(r)

# 现在可以正常使用了
img_054_1 = [r for r in results_with_folder if r['folder'] == '054_brain'][0]      # 第1张
img_054_12 = [r for r in results_with_folder if r['folder'] == '054_brain'][11]    # 第12张
img_079_9 = [r for r in results_with_folder if r['folder'] == '079_brain'][8]      # 第9张
img_079_25 = [r for r in results_with_folder if r['folder'] == '079_brain'][24]    # 第25张

selected = [
    ('054_brain #1', img_054_1),
    ('054_brain #12', img_054_12),
    ('079_brain #9', img_079_9),
    ('079_brain #25', img_079_25)
]

# 创建4x3网格显示
fig, axes = plt.subplots(4, 3, figsize=(14, 18))

for i, (title, sample) in enumerate(selected):
    # 原始图像
    axes[i, 0].imshow(sample['img_orig'], cmap='gray')
    axes[i, 0].set_title(f'{title}\nOriginal', fontsize=11)
    axes[i, 0].axis('off')

    # DCT去噪
    axes[i, 1].imshow(sample['img_dct'], cmap='gray')
    axes[i, 1].set_title(f'DCT\nPSNR={sample["psnr_dct"]:.1f} dB\nSSIM={sample["ssim_dct"]:.3f}', fontsize=10)
    axes[i, 1].axis('off')

    # 高斯滤波
    axes[i, 2].imshow(sample['img_gauss'], cmap='gray')
    axes[i, 2].set_title(f'Gaussian\nPSNR={sample["psnr_gauss"]:.1f} dB\nSSIM={sample["ssim_gauss"]:.3f}', fontsize=10)
    axes[i, 2].axis('off')

plt.suptitle('DCT vs Gaussian Filtering: Brain CT Denoising Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 打印详细对比信息
print("\n" + "="*70)
print("详细对比结果")
print("="*70)

step9. 可视化：所有图片的 PSNR/SSIM 对比柱状图

In [ ]:
names = [r['name'] for r in results]
psnr_dct = [r['psnr_dct'] for r in results]
psnr_gauss = [r['psnr_gauss'] for r in results]
ssim_dct = [r['ssim_dct'] for r in results]
ssim_gauss = [r['ssim_gauss'] for r in results]

x = np.arange(len(names))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# PSNR 对比
ax1.bar(x - width/2, psnr_dct, width, label='DCT')
ax1.bar(x + width/2, psnr_gauss, width, label='Gaussian')
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('PSNR Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=45, ha='right')
ax1.legend()
# SSIM 对比
ax2.bar(x - width/2, ssim_dct, width, label='DCT')
ax2.bar(x + width/2, ssim_gauss, width, label='Gaussian')
ax2.set_ylabel('SSIM')
ax2.set_title('SSIM Comparison')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=45, ha='right')
ax2.legend()

plt.tight_layout()
plt.show()